# Experiment Analysis Template

Use this notebook to deep-dive into a specific training run logged in MLflow.
You just need to provide the `RUN_ID`.

In [ ]:
# === CONFIGURATION ===
# Copy this from the MLflow UI (Experiment -> Run -> Artifacts/Details)
RUN_ID = "YOUR_RUN_ID_HERE"
MLFLOW_TRACKING_URI = "mlruns" # or http://localhost:5000

In [ ]:
import mlflow
import pandas as pd
import polars as pl
import shap
import matplotlib.pyplot as plt
import xgboost as xgb
from src.data.feature_store import FeatureStore
from src.data.schema import FEATURE_SCHEMA

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

## 1. Load Model and Metrics

In [ ]:
# Load Run Info
run = mlflow.get_run(RUN_ID)
params = run.data.params
metrics = run.data.metrics

print(f"Experiment: {run.info.experiment_id}")
print(f"Variant: {params.get('model.variant', 'Unknown')}")
print("Top Metrics:")
for k, v in metrics.items():
    if "mean" in k:
        print(f"  {k}: {v:.4f}")

In [ ]:
# Load Model Artifact
# Note: The accumulated pipeline often saves the XGBoost model as 'xgboost_model'
model_uri = f"runs:/{RUN_ID}/xgboost_model"
try:
    model = mlflow.xgboost.load_model(model_uri)
    print("Model loaded successfully.")
except Exception as e:
    print(f"Could not load model. Ensure it was logged properly. Error: {e}")

## 2. SHAP Analysis (Feature Importance)

In [ ]:
if model:
    # We need some dummy data or test data to compute SHAP
    # Ideally, we load the same data used for test.
    # For this template, we'll try to load a small sample from the Feature Store
    # Or you can assume the model has internal feature names
    
    # Plot Feature Importance (Gain)
    xgb.plot_importance(model, max_num_features=20)
    plt.title("XGBoost Feature Importance (Weight)")
    plt.show()

## 3. Performance Over Time (Drift Check)

In [ ]:
# Filter metrics for fold-specific performance
fold_metrics = {}
for k, v in metrics.items():
    if "fold_" in k and "auc_pr" in k:
        # Format fold_X_auc_pr
        parts = k.split("_")
        fold_idx = int(parts[1])
        fold_metrics[fold_idx] = v

if fold_metrics:
    folds = sorted(fold_metrics.keys())
    values = [fold_metrics[f] for f in folds]
    
    plt.plot(folds, values, marker='o')
    plt.xlabel("Split Index (Time)")
    plt.ylabel("AUC-PR")
    plt.title("Performance Stability Over Time")
    plt.ylim(0, 1.0)
    plt.grid(True)
    plt.show()